# Tuần 2 — PhoBERT Multi-task Training
**ABSA VLSP 2018 Hotel | NLP Course — HUST**

**Môi trường:** Kaggle Notebook (T4 GPU) — kéo code từ GitHub

| Cell | Mô tả |
|------|-------|
| 1 | Check GPU + Install dependencies |
| 2 | **Clone repo từ GitHub + Setup** |
| 3 | Download dataset VLSP 2018 |
| 4 | Preprocessing (dùng cache nếu có) |
| 5 | Kiểm tra EDA config |
| **6** | **TRAIN — concat_4_layers (SOTA)** |
| 7 | Ablation: cls_only |
| 8 | Learning curve |
| 9 | Summary report + copy kết quả |

> ⚠️ Chạy theo thứ tự từ trên xuống. Không skip cell nào.

In [ ]:
# ============================================================
# Cell 1 — Check GPU & Install dependencies
# ============================================================
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu}")
    print(f"   VRAM: {vram:.1f} GB")
    if vram < 14:
        print("⚠️  VRAM < 14GB — nếu OOM thì giảm batch_size về 4 trong constants.py")
else:
    raise RuntimeError("❌ Không có GPU! Kaggle: Settings → Accelerator → GPU T4 x2")

torch.cuda.empty_cache()
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

# Install packages
!pip install -q transformers==4.38.0 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece
print("✅ Dependencies installed")

In [ ]:
# ============================================================
# Cell 2 — Clone repo từ GitHub & Setup working directory
# ============================================================
import os, sys

REPO_URL    = "https://github.com/vudinhminh08/NLP-project-master-study.git"
REPO_BRANCH = "master"
PROJECT_DIR = "/kaggle/working/absa-project"

# Clone (hoặc pull nếu đã có)
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print("✅ Clone xong")
else:
    print(f"Repo đã tồn tại tại {PROJECT_DIR} — pulling latest...")
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

# Chuyển vào thư mục project
os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

# Tạo thư mục cần thiết
for d in ["data", "outputs/models", "outputs/results", "outputs/eda"]:
    os.makedirs(d, exist_ok=True)

# Thêm Python paths
sys.path.insert(0, "code/week1")
sys.path.insert(0, "code/week2")

# Kiểm tra cấu trúc repo
print("\nCấu trúc repo:")
!ls -la
!ls code/week1/ code/week2/

In [ ]:
# ============================================================
# Cell 3 — Download dataset VLSP 2018
# ============================================================
import pandas as pd, os

if not os.path.exists("data/train.csv"):
    print("Downloading VLSP 2018 Hotel dataset...")
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/
    print("✅ Data downloaded")
else:
    print("✅ Data đã tồn tại")

for split in ["train", "dev", "test"]:
    df = pd.read_csv(f"data/{split}.csv")
    print(f"  {split}: {len(df)} rows × {df.shape[1]} cols")

In [ ]:
# ============================================================
# Cell 4 — Preprocessing
# ============================================================
import pandas as pd, os

FORCE_REPROCESS = False

if (not FORCE_REPROCESS) and os.path.exists("data/train_preprocessed.csv"):
    print("✅ Cache đã có (data/*_preprocessed.csv)")
    s = pd.read_csv("data/train_preprocessed.csv").iloc[0]
    print(f"  Original : {s['Review'][:80]}")
    print(f"  Processed: {str(s.get('processed_review', 'N/A'))[:80]}")
else:
    print("Chưa có cache — chạy preprocessing với VnCoreNLP...")
    from step3_preprocessing import preprocess_dataframe, VnCoreNLPSegmenter
    import py_vncorenlp
    vncorenlp_dir = os.path.join(os.getcwd(), 'vncorenlp')
    if not os.path.exists(os.path.join(vncorenlp_dir, 'models', 'wordsegmenter', 'wordsegmenter.rdr')):
        print('Downloading VnCoreNLP models...')
        py_vncorenlp.download_model(save_dir=vncorenlp_dir)
    segmenter = VnCoreNLPSegmenter(vncorenlp_dir=vncorenlp_dir, use_fallback=False)
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(f"data/{split}.csv")
        preprocess_dataframe(df, segmenter=segmenter,
                             cache_path=f"data/{split}_preprocessed.csv")
        print(f"  ✅ {split}: {len(df)} rows processed")
    segmenter.close()
    print("✅ Preprocessing hoàn tất")

In [ ]:
# ============================================================
# Cell 5 — Kiểm tra EDA config & class weights
# ============================================================
import json, os

# Encoder config
enc_path = "outputs/eda/encoder_config.json"
if not os.path.exists(enc_path):
    raise FileNotFoundError(
        f"Không tìm thấy {enc_path}. "
        "EDA outputs phải có trong repo (git commit outputs/eda/)."
    )

enc_cfg = json.load(open(enc_path))
print("=== Encoder Config ===")
for k, v in enc_cfg.items():
    print(f"  {k}: {v}")

# Class weights
cw = json.load(open("outputs/eda/class_weights.json"))
print("\n=== Global Class Weights (trước khi clip) ===")
label_map = {"0": "absent", "1": "positive", "2": "negative", "3": "neutral"}
for cls, w in cw['global_weights'].items():
    note = " ← clip về 10.0" if float(w) > 10 else ""
    print(f"  {label_map.get(cls,cls):12s}: {float(w):.1f}x{note}")

# Train config
from utils.constants import TRAIN_CONFIG, ZERO_TRAIN_ASPECTS
print("\n=== Train Config ===")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")
print(f"\n  ZERO_TRAIN_ASPECTS (excluded from Macro-F1): {ZERO_TRAIN_ASPECTS}")
print(f"\n✅ Config OK — seq_len={enc_cfg['recommended_max_seq_len']}, encoder={enc_cfg['encoder_option']}")

In [ ]:
# ============================================================
# Cell 6 — TRAIN: concat_4_layers (SOTA architecture)
# ============================================================
# concat_4_layers: concat 4 hidden layers cuối tại [CLS] → 3072 dim
# - Mixed Precision (AMP) tự động bật trên GPU → training ~2x nhanh hơn
# - Early stopping patience=3 dựa trên dev Combined F1
# - Lưu best_model.pt sau mỗi epoch có cải thiện
# Expected: 5-10 epochs, ~20-40 phút trên T4

import torch
torch.cuda.empty_cache()
print(f"VRAM free before training: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB")

from run_experiment import main

test_metrics = main(encoder_option="concat_4_layers", use_amp=True)

print("\n" + "="*55)
print("MAIN RUN RESULTS")
print(f"  ACD F1:      {test_metrics['macro_acd_f1']:.4f}  (SOTA: 0.8255)")
print(f"  SPC F1:      {test_metrics['macro_spc_f1']:.4f}")
print(f"  Combined F1: {test_metrics['macro_combined_f1']:.4f}  (SOTA: 0.7732)")
print("="*55)

In [ ]:
# ============================================================
# Cell 7 — Ablation: cls_only
# ============================================================
# Chứng minh kỹ thuật concat 4 layers có đóng góp thực sự
# (so sánh 3072 dim vs 768 dim trong báo cáo)
# Chạy SAU khi Cell 6 đã hoàn tất

import torch
torch.cuda.empty_cache()

from run_experiment import main

test_metrics_cls = main(encoder_option="cls_only", use_amp=True)

print("\n" + "="*55)
print("ABLATION SUMMARY")
print(f"  concat_4_layers: {test_metrics['macro_combined_f1']:.4f}  ← SOTA architecture")
print(f"  cls_only:        {test_metrics_cls['macro_combined_f1']:.4f}")
gain = test_metrics['macro_combined_f1'] - test_metrics_cls['macro_combined_f1']
print(f"  Gain từ concat:  {gain*100:+.2f}%  {'✅ concat tốt hơn' if gain > 0 else '⚠️ cls_only bằng hoặc tốt hơn'}")
print("="*55)

In [ ]:
# ============================================================
# Cell 8 — Learning Curve (BẮT BUỘC cho báo cáo)
# ============================================================
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = json.load(open("outputs/results/training_history.json"))
best_ep = history["best_epoch"]
epochs  = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("PhoBERT concat_4_layers — Learning Curve (ABSA VLSP 2018)", fontsize=13)

# --- Loss ---
ax1.plot(epochs, history["train_loss"], "o-", c="crimson",   lw=2, label="Train Loss")
ax1.plot(epochs, history["dev_loss"],   "o-", c="steelblue", lw=2, label="Dev Loss")
ax1.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax1.set(title="Loss", xlabel="Epoch", ylabel="Cross-Entropy Loss")
ax1.legend(); ax1.grid(alpha=0.3)

# --- F1 ---
ax2.plot(epochs, history["dev_acd_f1"],      "s-", c="darkorange", lw=2, label="Dev ACD F1")
ax2.plot(epochs, history["dev_spc_f1"],      "^-", c="purple",     lw=2, label="Dev SPC F1")
ax2.plot(epochs, history["dev_combined_f1"], "o-", c="green",      lw=2.5, label="Dev Combined F1")
ax2.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax2.axhline(0.7732,  c="red",   ls=":",  alpha=0.5, label="SOTA Combined 0.7732")
ax2.set(title="F1 Score", xlabel="Epoch", ylabel="Macro F1")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/eda/learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Phân tích cho báo cáo ---
print(f"\n📊 Phân tích learning curve (cho báo cáo):")
print(f"  Best epoch:           {best_ep} / {len(history['train_loss'])}")
print(f"  Best Combined F1:     {history['best_combined_f1']:.4f}")

if len(history["train_loss"]) > best_ep:
    tloss_at   = history["train_loss"][best_ep - 1]
    tloss_after = history["train_loss"][best_ep]
    dloss_at   = history["dev_loss"][best_ep - 1]
    dloss_after = history["dev_loss"][best_ep]
    print(f"  Train loss epoch {best_ep}: {tloss_at:.4f} → epoch {best_ep+1}: {tloss_after:.4f} (tiếp tục giảm)")
    print(f"  Dev loss epoch {best_ep}:   {dloss_at:.4f} → epoch {best_ep+1}: {dloss_after:.4f} (tăng = overfit)")
    print(f"  → Early stopping đúng: dev F1 không cải thiện {history['config']['early_stop_patience']} epoch liên tiếp")

print(f"\n  Saved: outputs/eda/learning_curve.png")

In [ ]:
# ============================================================
# Cell 9 — Summary Report & Copy kết quả
# ============================================================
import os, json, shutil

# In summary report
report = "outputs/results/week2_summary.md"
if os.path.exists(report):
    print(open(report, encoding="utf-8").read())
else:
    print("Chưa có report — đảm bảo Cell 6 đã chạy xong")

# Liệt kê tất cả files kết quả
print("\n=== Kết quả đã tạo ===")
result_files = []
for root, dirs, files in os.walk("outputs"):
    for f in files:
        if not f.endswith(".DS_Store"):
            path = os.path.join(root, f)
            size = os.path.getsize(path)
            result_files.append(path)
            print(f"  {path} ({size/1024:.1f} KB)")

# Tạo zip để download
print("\nTạo zip...")
shutil.make_archive("/kaggle/working/week2_results", "zip", "outputs")
print("✅ Zip: /kaggle/working/week2_results.zip")
print("   → Kaggle: Output panel → Download")

# Copy summary để dễ copy-paste
print("\n=== Số liệu quan trọng để điền vào SESSION_HANDOFF.md ===")
if os.path.exists("outputs/results/week2_test_metrics.json"):
    m = json.load(open("outputs/results/week2_test_metrics.json"))
    print(f"__WEEK2_ACD_F1__      = {m['macro_acd_f1']:.4f}")
    print(f"__WEEK2_SPC_F1__      = {m['macro_spc_f1']:.4f}")
    print(f"__WEEK2_COMBINED_F1__ = {m['macro_combined_f1']:.4f}")
if os.path.exists("outputs/results_cls_only/week2_test_metrics.json"):
    m2 = json.load(open("outputs/results_cls_only/week2_test_metrics.json"))
    print(f"__WEEK2_ABLATION_COMBINED_F1__ = {m2['macro_combined_f1']:.4f}")